# 03 — Baseline Response Model and Why It Fails

A response model predicts conversion probability.

That is useful, but it does not answer whether the campaign caused the conversion.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
import numpy as np

from uplift_marketing.data import generate_synthetic_uplift_data

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from uplift_marketing.features import infer_feature_spec, build_preprocessor, split_features_treatment_outcome

data = generate_synthetic_uplift_data(n_rows=5_000, random_state=42)
spec = infer_feature_spec(data, extra_excluded=["customer_id", "visit"])
x, treatment, y = split_features_treatment_outcome(data, spec)

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42, stratify=y)
model = Pipeline([
    ("preprocessor", build_preprocessor(spec)),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
model.fit(x_train, y_train)
response_score = model.predict_proba(x_test)[:, 1]
roc_auc_score(y_test, response_score)


## Why this is not enough

A high response score can identify customers likely to convert. But some of those customers may convert without treatment.

For campaign allocation, the more relevant quantity is the difference between predicted treated and control outcomes.
